In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_brands = spark.table('ecommerce.silver.slv_brands')
df_category = spark.table('ecommerce.silver.slv_category')
df_products = spark.table('ecommerce.silver.slv_products')

In [0]:
df_brands.createOrReplaceTempView('tbl_brands')
df_category.createOrReplaceTempView('tbl_category')
df_products.createOrReplaceTempView('tbl_products')


In [0]:
%%sql
USE CATALOG ecommerce

In [0]:
%sql
CREATE OR REPLACE TABLE gold.gld_dim_products AS 
(
WITH brandxcategory AS (
SELECT 
c.category_name,
c.category_code,
b.brand_name,
b.brand_code
FROM tbl_brands b
INNER JOIN tbl_category c
ON b.category_code = c.category_code
)

SELECT 
p.product_id,
  p.sku,
  p.category_code,
  COALESCE(bc.category_name, 'Not Available') AS category_name,
  p.brand_code,
  COALESCE(bc.brand_name, 'Not Available')   AS brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.file_name,
  p.ingest_timestamp

FROM
 tbl_products p
LEFT JOIN brandxcategory bc
ON p.brand_code = bc.brand_code
)

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
country_state_map

In [0]:
# 1 Flatten country_state_map into a list of Rows
rows = []
for country, states in country_state_map.items(): 
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]        

In [0]:
df_region_mapping = spark.createDataFrame(rows)

df_region_mapping.show()

In [0]:
df_silver = spark.table('ecommerce.silver.slv_customer')

In [0]:
df_gold = df_silver.join(df_region_mapping, ["country", "state"], "left")
df_gold = df_gold.fillna({'region': 'Other'})

display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.gold.gld_dim_customers")


In [0]:
df_silver = spark.table('ecommerce.silver.slv_date')
display(df_silver)

In [0]:
df_gold = df_silver.withColumn("date_id", date_format("date", 'ddMMyyyy').cast(StringType()))

df_gold = df_gold.withColumn("month_name", date_format("date", 'MMMM'))

df_gold = df_gold.withColumn("is_weekend", when(col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0))

desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "ingested_at", "_source_file"]

df_gold = df_gold.select(desired_columns_order)

display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.gold.gld_dim_date")
